# Results for model: microsoft/phi-3-small-8k-instruct

In [ ]:
 def create_global_top_quantile_feature(df, feature_col, responder_col, batch_size):
      global_top_quantile = []
      for i in range(0, len(df), batch_size):
          batch_df = df[i:i+batch_size]
          top_quantile_value = batch_df[batch_df[feature_col] == batch_df[feature_col].quantile(0.85)][feature_col].first()
          global_top_quantile.extend(batch_df[feature_col] == top_quantile_value)
      return global_top_quantile

  # Feature Engineering
  train_df = pl.read_parquet(DATA_PATH)
  global_top_quantile_feature = create_global_top_quantile_feature(
      train_df, 'feature_00', 'responder_6', 1000
  )
  train_df = train_df.with_column(pl.col('global_top_quantile').fill(0).cast(pl.UInt8))
  train_df = train_df.drop('feature_00')

  # Train XGBoost Regressor
  xgb_reg = xgboost.XGBRegressor(objective='reg:squarederror', n_estimators=1000)
  xgb_reg.fit(train_df.select(['global_top_quantile', 'responder_6']).collect(), 
              train_df['responder_6'].collect())

  # Predictions
  test_df = pl.read_parquet(DATA_PATH)
  test_df = test_df.with_column(pl.lit(xgb_reg.predict(test_df.select(['global_top_quantile']).collect())))
  test_df = test_df.drop('global_top_quantile')